# 🚀 End-to-End AI Pipeline: Instagram → Risk Analysis → Case Management

This notebook executes the complete pipeline from raw social media data to operational case management.

## Pipeline Stages

```
📥 Instagram Data (164 posts, 11 users)
    ↓
🤖 Stage 1: NLP Signal Extraction
    • Emotion detection (distress, anger, fear)
    • Sentiment analysis (positive/negative valence)
    • Harm risk assessment (self-harm, violence)
    ↓
📊 Stage 2: Behavioral Feature Engineering
    • Aggregate signals per user
    • Compute behavioral metrics
    • Multi-dimensional risk scoring
    ↓
⚙️  Stage 3: Case Promotion
    • Create/update operational cases
    • Track risk history
    • Generate checklists
    ↓
👥 Youth Workers (Dashboard)
```

## Database Architecture (Option A)

**instagram_scraper** (Analytics Truth):
- `instagram_posts`, `instagram_users` (source data)
- `text_units_signals` (Stage 1 output)
- `case_risk_profiles` (Stage 2 output)

**dellinnovate** (Operational Truth):
- `scs_cases` (case records)
- `scs_case_history` (risk evolution)
- `scs_checklist` (task tracking)

## Setup Environment

In [1]:
import os
import sys
import asyncio
from datetime import datetime
from pprint import pprint
import nest_asyncio

# Enable async/await in Jupyter
nest_asyncio.apply()

# Add backend to path
repo_root = os.path.dirname(os.getcwd())
backend_path = os.path.join(repo_root, 'backend')
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

print(f"✅ Backend path added: {backend_path}")
print(f"📁 Current directory: {os.getcwd()}")

# Enable autoawait
try:
    get_ipython().run_line_magic('autoawait', 'True')
    print("✅ Async support enabled")
except:
    print("ℹ️  Running in standard Python mode")

✅ Backend path added: /Users/pandeymahi/Documents/GitHub/DellInnovate2026_Team-Untitled/backend
📁 Current directory: /Users/pandeymahi/Documents/GitHub/DellInnovate2026_Team-Untitled/demos
✅ Async support enabled


## Step 1: Connect to Databases & Check Initial State

In [2]:
from config.database import MongoDB

async def check_initial_state():
    """Connect to databases and show initial state"""
    await MongoDB.connect_db()
    source_db = MongoDB.get_source_db()  # instagram_scraper
    db = MongoDB.get_db()  # dellinnovate
    
    print("=" * 70)
    print("📊 INITIAL DATABASE STATE")
    print("=" * 70)
    
    print("\n📥 SOURCE & ANALYTICS (instagram_scraper):")
    posts_count = await source_db.instagram_posts.count_documents({})
    users_count = await source_db.instagram_users.count_documents({})
    signals_count = await source_db.text_units_signals.count_documents({})
    profiles_count = await source_db.case_risk_profiles.count_documents({})
    print(f"  Posts: {posts_count:,} (input data)")
    print(f"  Users: {users_count:,}")
    print(f"  NLP Signals: {signals_count:,} (Stage 1 output)")
    print(f"  Risk Profiles: {profiles_count:,} (Stage 2 output)")
    
    print("\n⚙️  OPERATIONAL (dellinnovate):")
    cases_count = await db.scs_cases.count_documents({})
    history_count = await db.scs_case_history.count_documents({})
    checklist_count = await db.scs_checklist.count_documents({})
    print(f"  Cases: {cases_count:,} (Stage 3 output)")
    print(f"  Case History: {history_count:,}")
    print(f"  Checklist Items: {checklist_count:,}")
    
    print("\n" + "=" * 70)
    
    return {
        'posts': posts_count,
        'users': users_count,
        'signals_before': signals_count,
        'profiles_before': profiles_count,
        'cases_before': cases_count,
        'history_before': history_count,
        'checklist_before': checklist_count
    }

initial_state = await check_initial_state()

2026-03-07 21:00:22.078 | SUCCESS  | config.database:connect_db:33 - Connected to MongoDB
2026-03-07 21:00:22.079 | INFO     | config.database:connect_db:34 -   Primary DB (analytics): dellinnovate
2026-03-07 21:00:22.080 | INFO     | config.database:connect_db:35 -   Source DB (Instagram): instagram_scraper


📊 INITIAL DATABASE STATE

📥 SOURCE & ANALYTICS (instagram_scraper):
  Posts: 164 (input data)
  Users: 11
  NLP Signals: 450 (Stage 1 output)
  Risk Profiles: 21 (Stage 2 output)

⚙️  OPERATIONAL (dellinnovate):
  Cases: 21 (Stage 3 output)
  Case History: 84
  Checklist Items: 84



In [ ]:
async def clear_pipeline_outputs():
    """Delete all pipeline outputs while preserving source data"""
    
    source_db = MongoDB.get_source_db()
    db = MongoDB.get_db()
    
    print("=" * 70)
    print("🗑️  CLEARING PIPELINE OUTPUTS")
    print("=" * 70)
    
    # Show current counts
    print("\n📊 Current state:")
    signals_count = await source_db.text_units_signals.count_documents({})
    profiles_count = await source_db.case_risk_profiles.count_documents({})
    cases_count = await db.scs_cases.count_documents({})
    history_count = await db.scs_case_history.count_documents({})
    checklist_count = await db.scs_checklist.count_documents({})
    
    print(f"  NLP Signals (Stage 1): {signals_count:,}")
    print(f"  Risk Profiles (Stage 2): {profiles_count:,}")
    print(f"  Cases (Stage 3): {cases_count:,}")
    print(f"  Case History: {history_count:,}")
    print(f"  Checklist Items: {checklist_count:,}")
    
    # Confirm deletion
    print("\n⚠️  This will delete:")
    print("   • text_units_signals (instagram_scraper)")
    print("   • case_risk_profiles (instagram_scraper)")
    print("   • scs_cases (dellinnovate)")
    print("   • scs_case_history (dellinnovate)")
    print("   • scs_checklist (dellinnovate)")
    print("\n✅ Source data will be preserved:")
    print("   • instagram_posts (164 posts)")
    print("   • instagram_users (11 users)")
    
    # Delete collections
    print("\n🗑️  Deleting collections...")
    
    result_signals = await source_db.text_units_signals.delete_many({})
    print(f"  ✅ Deleted {result_signals.deleted_count:,} signals")
    
    result_profiles = await source_db.case_risk_profiles.delete_many({})
    print(f"  ✅ Deleted {result_profiles.deleted_count:,} risk profiles")
    
    result_cases = await db.scs_cases.delete_many({})
    print(f"  ✅ Deleted {result_cases.deleted_count:,} cases")
    
    result_history = await db.scs_case_history.delete_many({})
    print(f"  ✅ Deleted {result_history.deleted_count:,} history entries")
    
    result_checklist = await db.scs_checklist.delete_many({})
    print(f"  ✅ Deleted {result_checklist.deleted_count:,} checklist items")
    
    # Verify
    print("\n📊 Final state:")
    print(f"  NLP Signals: {await source_db.text_units_signals.count_documents({})}")
    print(f"  Risk Profiles: {await source_db.case_risk_profiles.count_documents({})}")
    print(f"  Cases: {await db.scs_cases.count_documents({})}")
    print(f"  Case History: {await db.scs_case_history.count_documents({})}")
    print(f"  Checklist Items: {await db.scs_checklist.count_documents({})}")
    
    print("\n" + "=" * 70)
    print("✅ Pipeline outputs cleared! Ready to run from scratch.")
    print("=" * 70)

# Uncomment the line below to run cleanup
#await clear_pipeline_outputs()

🗑️  CLEARING PIPELINE OUTPUTS

📊 Current state:
  NLP Signals (Stage 1): 450
  Risk Profiles (Stage 2): 21
  Cases (Stage 3): 21
  Case History: 84
  Checklist Items: 84

⚠️  This will delete:
   • text_units_signals (instagram_scraper)
   • case_risk_profiles (instagram_scraper)
   • scs_cases (dellinnovate)
   • scs_case_history (dellinnovate)
   • scs_checklist (dellinnovate)

✅ Source data will be preserved:
   • instagram_posts (164 posts)
   • instagram_users (11 users)

🗑️  Deleting collections...
  ✅ Deleted 450 signals
  ✅ Deleted 21 risk profiles
  ✅ Deleted 21 cases
  ✅ Deleted 84 history entries
  ✅ Deleted 84 checklist items

📊 Final state:
  NLP Signals: 0
  Risk Profiles: 0
  Cases: 0
  Case History: 0
  Checklist Items: 0

✅ Pipeline outputs cleared! Ready to run from scratch.


## 🗑️ Optional: Clear Pipeline Outputs

Use this cell to reset the pipeline and delete all generated analytics and operational data. **Source data (posts, users) will be preserved.**

## Step 2: Stage 1 - NLP Signal Extraction

Extract psychological signals from each post:
- **Emotion Detection**: Distress, anger, fear, sadness
- **Sentiment Analysis**: Positive/negative valence
- **Harm Risk**: Self-harm indicators, violence markers

In [4]:
from analytics.signal_extraction import NLPSignalExtractor

async def run_stage1_signal_extraction():
    """Run Stage 1: Extract NLP signals from posts"""
    
    print("=" * 70)
    print("🤖 STAGE 1: NLP SIGNAL EXTRACTION")
    print("=" * 70)
    
    source_db = MongoDB.get_source_db()
    
    # Initialize extractor
    extractor = NLPSignalExtractor()
    
    print("\n🚀 Running complete NLP pipeline...")
    print("   • Step 1: Extract text units from posts")
    print("   • Step 2: Preprocess text")
    print("   • Step 3: Run NLP models (emotion, sentiment, harm)")
    print("   • Step 4: Create signal documents")
    print("   • Step 5: Store to database")
    
    # Run the complete pipeline
    results = await extractor.run_pipeline(
        case_users=None,  # Process all users
        limit=None,       # Process all posts
        export_csv=False  # Skip CSV export
    )
    
    print(f"\n✅ Pipeline Complete!")
    print(f"   Text units processed: {results['text_units_found']}")
    print(f"   Valid units analyzed: {results['valid_units']}")
    print(f"   Signals stored: {results['signals_created']}")
    print(f"   Duration: {results['duration_seconds']:.2f} seconds")
    
    # Show sample signal
    signals = await source_db.text_units_signals.find().limit(1).to_list(length=1)
    if signals:
        sample = signals[0]
        print("\n📊 Sample Signal:")
        print(f"   User: {sample['case_user']}")
        print(f"   Text: {sample['text'][:80]}...")
        print(f"   Emotion Score: {sample.get('emotion_score', 0):.3f}")
        print(f"   Sentiment Score: {sample.get('sentiment_score', 0):.3f}")
        print(f"   Harm Score: {sample.get('harm_score', 0):.3f}")
    
    print("\n" + "=" * 70)
    return results

signals = await run_stage1_signal_extraction()

2026-03-07 21:00:51.340 | INFO     | analytics.signal_extraction:__init__:56 - NLP Signal Extractor initialized
2026-03-07 21:00:51.340 | INFO     | analytics.signal_extraction:run_pipeline:318 - ======================================================================
2026-03-07 21:00:51.340 | INFO     | analytics.signal_extraction:run_pipeline:319 - Starting NLP Signal Extraction Pipeline (Stage 1)
2026-03-07 21:00:51.340 | INFO     | analytics.signal_extraction:run_pipeline:320 - ======================================================================
2026-03-07 21:00:51.341 | INFO     | analytics.signal_extraction:run_pipeline:323 - [1/5] Extracting text units from database...


🤖 STAGE 1: NLP SIGNAL EXTRACTION

🚀 Running complete NLP pipeline...
   • Step 1: Extract text units from posts
   • Step 2: Preprocess text
   • Step 3: Run NLP models (emotion, sentiment, harm)
   • Step 4: Create signal documents
   • Step 5: Store to database


2026-03-07 21:00:51.799 | INFO     | analytics.signal_extraction:extract_text_units_from_db:93 - Retrieved 164 posts from source database (instagram_scraper)
2026-03-07 21:00:51.800 | INFO     | analytics.signal_extraction:extract_text_units_from_db:101 - Extracted 326 text units
2026-03-07 21:00:51.800 | INFO     | analytics.signal_extraction:run_pipeline:336 - [2/5] Preprocessing text units...
2026-03-07 21:00:51.807 | INFO     | analytics.signal_extraction:run_pipeline:341 - Valid text units: 225/326
2026-03-07 21:00:51.808 | INFO     | analytics.signal_extraction:run_pipeline:344 - [3/5] Running NLP analysis (sentiment, emotion, distortion)...
2026-03-07 21:00:51.808 | INFO     | analytics.signal_extraction:_load_nlp_pipeline:61 - Loading NLP models...
2026-03-07 21:00:51.809 | INFO     | analytics.nlp_models:__init__:373 - Initializing NLP Pipeline...
2026-03-07 21:00:51.809 | INFO     | analytics.nlp_models:__init__:33 - Loading sentiment model: cardiffnlp/twitter-roberta-base-se

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-07 21:00:54.394 | SUCCESS  | analytics.nlp_models:__init__:45 - Sentiment model loaded on cpu
2026-03-07 21:00:54.395 | INFO     | analytics.nlp_models:__init__:131 - Loading emotion model: j-hartmann/emotion-english-distilroberta-base


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-07 21:00:56.201 | SUCCESS  | analytics.nlp_models:__init__:143 - Emotion model loaded on cpu
2026-03-07 21:00:56.201 | INFO     | analytics.nlp_models:__init__:278 - Loading distortion detection model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-07 21:01:01.254 | SUCCESS  | analytics.nlp_models:__init__:289 - Cognitive distortion detector initialized
2026-03-07 21:01:01.259 | SUCCESS  | analytics.nlp_models:__init__:379 - NLP Pipeline ready
2026-03-07 21:01:01.259 | SUCCESS  | analytics.signal_extraction:_load_nlp_pipeline:63 - NLP models loaded
2026-03-07 21:01:07.422 | INFO     | analytics.signal_extraction:run_pipeline:347 - Processed 50/225 text units...
2026-03-07 21:01:11.749 | INFO     | analytics.signal_extraction:run_pipeline:347 - Processed 100/225 text units...
2026-03-07 21:01:16.896 | INFO     | analytics.signal_extraction:run_pipeline:347 - Processed 150/225 text units...
2026-03-07 21:01:21.81


✅ Pipeline Complete!
   Text units processed: 326
   Valid units analyzed: 225
   Signals stored: 225
   Duration: 33.72 seconds

📊 Sample Signal:
   User: chrishemsworth
   Text: I’m launching a new series with my best mates! I’m sending @azzagrist and @zocob...
   Emotion Score: 0.886
   Sentiment Score: 0.965
   Harm Score: 0.000



## Step 3: Stage 2 - Feature Engineering & Risk Scoring

Aggregate per-text signals into per-user risk profiles:
- **Behavioral Metrics**: Distortion, sentiment trends, emotion patterns, engagement
- **Risk Calculation**: Multi-dimensional scoring (0-100 scale)
- **Risk Classification**: High/Medium/Low
- **Evidence Extraction**: Sample high-risk content for context

In [5]:
from analytics.feature_engineering import BehavioralFeatureEngineer

async def run_stage2_feature_engineering():
    """Run Stage 2: Aggregate signals into risk profiles"""
    
    print("=" * 70)
    print("📊 STAGE 2: FEATURE ENGINEERING & RISK SCORING")
    print("=" * 70)
    
    source_db = MongoDB.get_source_db()
    
    # Check input data
    signals_count = await source_db.text_units_signals.count_documents({})
    print(f"\n📥 Input: {signals_count:,} signals from Stage 1")
    
    if signals_count == 0:
        print("❌ No signals found! Run Stage 1 first.")
        return None
    
    # Initialize engineer
    print("\n🔧 Initializing behavioral feature engineering...")
    engineer = BehavioralFeatureEngineer()
    
    print("\n🚀 Running complete feature engineering pipeline...")
    print("   • Step 1: Get unique users from signals")
    print("   • Step 2: Fetch signals for each user")
    print("   • Step 3: Compute behavioral metrics")
    print("   • Step 4: Calculate risk scores")
    print("   • Step 5: Store risk profiles to database")
    
    # Run the complete pipeline
    results = await engineer.run_pipeline(
        case_users=None,  # Process all users
        window_days=7     # Use 7-day window
    )
    
    print(f"\n✅ Pipeline Complete!")
    print(f"   Users processed: {results['case_users_processed']}")
    print(f"   Profiles created: {results['profiles_created']}")
    print(f"   High-risk cases: {len(results['high_risk_cases'])}")
    print(f"   Duration: {results['duration_seconds']:.2f} seconds")
    
    if results['high_risk_cases']:
        print(f"\n⚠️  High-risk users: {', '.join(results['high_risk_cases'])}")
    
    # Show sample profile
    profiles = await source_db.case_risk_profiles.find().limit(1).to_list(length=1)
    if profiles:
        sample = profiles[0]
        # Map priority number to string (1=High, 2=Medium, 3=Low)
        priority_map = {1: "HIGH", 2: "MEDIUM", 3: "LOW"}
        priority_str = priority_map.get(sample.get('priority'), 'N/A')
        
        print("\n📊 Sample Risk Profile:")
        print(f"   User: {sample['case_user']}")
        print(f"   Risk Score: {sample['risk_score']:.1f}/100")
        print(f"   Risk Level: {sample['risk_level'].upper()}")
        print(f"   Priority: {priority_str} (P{sample.get('priority', 'N/A')})")
        print(f"   Text Units Analyzed: {sample.get('total_text_units', 0)}")
        print(f"   Distortion Rate: {sample.get('distortion_rate', 0):.2%}")
        print(f"   Avg Sentiment: {sample.get('avg_sentiment_score', 0):.3f}")
        print(f"   Distress Rate: {sample.get('distress_emotion_rate', 0):.2%}")
        if sample.get('top_distress_comments'):
            print(f"   Evidence Samples: {len(sample['top_distress_comments'])} high-risk texts")
    
    # Verify storage
    final_count = await source_db.case_risk_profiles.count_documents({})
    print(f"\n💾 Total profiles in database: {final_count:,}")
    
    # Show distribution
    print("\n📊 Risk Level Distribution:")
    pipeline = [
        {"$group": {"_id": "$risk_level", "count": {"$sum": 1}, "avg_score": {"$avg": "$risk_score"}}},
        {"$sort": {"avg_score": -1}}
    ]
    distribution = await source_db.case_risk_profiles.aggregate(pipeline).to_list(length=10)
    for item in distribution:
        print(f"   {item['_id'].upper()}: {item['count']} users (avg: {item['avg_score']:.1f})")
    
    print("\n" + "=" * 70)
    return results

profiles = await run_stage2_feature_engineering()

2026-03-07 21:01:25.257 | INFO     | analytics.feature_engineering:__init__:39 - Behavioral Feature Engineer initialized (using instagram_scraper)
2026-03-07 21:01:25.257 | INFO     | analytics.feature_engineering:run_pipeline:470 - ======================================================================
2026-03-07 21:01:25.258 | INFO     | analytics.feature_engineering:run_pipeline:471 - Starting Behavioral Feature Engineering Pipeline (Stage 2)
2026-03-07 21:01:25.258 | INFO     | analytics.feature_engineering:run_pipeline:472 - ======================================================================
2026-03-07 21:01:25.259 | INFO     | analytics.feature_engineering:run_pipeline:476 - Fetching all case users from signals...


📊 STAGE 2: FEATURE ENGINEERING & RISK SCORING

📥 Input: 225 signals from Stage 1

🔧 Initializing behavioral feature engineering...

🚀 Running complete feature engineering pipeline...
   • Step 1: Get unique users from signals
   • Step 2: Fetch signals for each user
   • Step 3: Compute behavioral metrics
   • Step 4: Calculate risk scores
   • Step 5: Store risk profiles to database


2026-03-07 21:01:25.326 | INFO     | analytics.feature_engineering:run_pipeline:479 - Processing 21 case users...
2026-03-07 21:01:25.326 | INFO     | analytics.feature_engineering:run_pipeline:486 - [1/21] Processing a24...
2026-03-07 21:01:25.327 | INFO     | analytics.feature_engineering:compute_risk_profile:360 - Computing risk profile for a24 (window: 7 days)
2026-03-07 21:01:25.405 | SUCCESS  | analytics.feature_engineering:compute_risk_profile:425 - Risk profile computed for a24: Score=7.3, Level=Low
2026-03-07 21:01:25.478 | INFO     | analytics.feature_engineering:store_risk_profile:452 - Risk profile stored: 69ac21a51ae5a3dc40778111
2026-03-07 21:01:25.479 | INFO     | analytics.feature_engineering:run_pipeline:486 - [2/21] Processing amazonalexa...
2026-03-07 21:01:25.480 | INFO     | analytics.feature_engineering:compute_risk_profile:360 - Computing risk profile for amazonalexa (window: 7 days)
2026-03-07 21:01:25.554 | SUCCESS  | analytics.feature_engineering:compute_risk_


✅ Pipeline Complete!
   Users processed: 21
   Profiles created: 21
   High-risk cases: 0
   Duration: 3.32 seconds

📊 Sample Risk Profile:
   User: a24
   Risk Score: 7.3/100
   Risk Level: LOW
   Priority: LOW (P3)
   Text Units Analyzed: 6
   Distortion Rate: 0.00%
   Avg Sentiment: 0.539
   Distress Rate: 0.00%
   Evidence Samples: 5 high-risk texts

💾 Total profiles in database: 21

📊 Risk Level Distribution:
   LOW: 21 users (avg: 17.8)



## Step 4: Stage 3 - Case Promotion to Operational System

Transform analytics into actionable cases:
- **Case Creation/Update**: Sync risk profiles to operational cases
- **History Tracking**: Record risk evolution over time
- **Checklist Generation**: Auto-create intervention tasks

In [ ]:
from services.case_promotion import promote_risk_profiles_to_scs_cases

async def run_stage3_case_promotion():
    """Run Stage 3: Promote risk profiles to operational cases"""
    
    print("=" * 70)
    print("⚙️  STAGE 3: CASE PROMOTION")
    print("=" * 70)
    
    source_db = MongoDB.get_source_db()
    db = MongoDB.get_db()
    
    # Check input data
    profiles_count = await source_db.case_risk_profiles.count_documents({})
    print(f"\n📥 Input: {profiles_count:,} risk profiles from Stage 2")
    
    if profiles_count == 0:
        print("❌ No risk profiles found! Run Stage 2 first.")
        return None
    
    # Check templates exist
    templates_count = await db.scs_checklist_templates.count_documents({})
    if templates_count == 0:
        print("\n⚠️  No checklist templates found. Creating defaults...")
        templates = [
            {"template_id": 1, "label": "Case Analysis Completed", "is_mandatory": True, "display_order": 1, "is_active": True, "created_at": datetime.utcnow()},
            {"template_id": 2, "label": "Outreach Attempted", "is_mandatory": True, "display_order": 2, "is_active": True, "created_at": datetime.utcnow()},
            {"template_id": 3, "label": "Response Received", "is_mandatory": True, "display_order": 3, "is_active": True, "created_at": datetime.utcnow()},
            {"template_id": 4, "label": "Follow-up Scheduled", "is_mandatory": True, "display_order": 4, "is_active": True, "created_at": datetime.utcnow()}
        ]
        for template in templates:
            await db.scs_checklist_templates.update_one(
                {"template_id": template["template_id"]},
                {"$set": template},
                upsert=True
            )
        print(f"✅ Created {len(templates)} templates")
    
    # Record before counts
    cases_before = await db.scs_cases.count_documents({})
    history_before = await db.scs_case_history.count_documents({})
    checklist_before = await db.scs_checklist.count_documents({})
    
    print("\n🚀 Running case promotion...")
    print("   • Reading from: instagram_scraper.case_risk_profiles")
    print("   • Writing to: dellinnovate.scs_cases")
    
    results = await promote_risk_profiles_to_scs_cases(
        db=db,
        min_priority="low",  # Process all priority levels
        limit=None,  # Process all profiles
        ingestion_timestamp=datetime.utcnow()
    )
    
    print("\n✅ Promotion Complete!\n")
    print("📊 Results:")
    print(f"   Profiles Read: {results['profiles_read']}")
    print(f"   Profiles Filtered: {results['profiles_filtered']}")
    print(f"   Cases Created: {results['cases_created']} 🆕")
    print(f"   Cases Updated: {results['cases_updated']} 🔄")
    print(f"   History Entries: {results['history_entries_added']}")
    print(f"   Checklist Items: {results['checklist_items_created']}")
    
    if results['errors']:
        print(f"\n⚠️  Errors: {len(results['errors'])}")
        for error in results['errors'][:3]:
            print(f"    - {error}")
    
    # Show changes
    cases_after = await db.scs_cases.count_documents({})
    history_after = await db.scs_case_history.count_documents({})
    checklist_after = await db.scs_checklist.count_documents({})
    
    print("\n📈 Database Changes:")
    print(f"   Cases: {cases_before} → {cases_after} (+{cases_after - cases_before})")
    print(f"   History: {history_before} → {history_after} (+{history_after - history_before})")
    print(f"   Checklist: {checklist_before} → {checklist_after} (+{checklist_after - checklist_before})")
    
    print("\n" + "=" * 70)
    return results

promotion_results = await run_stage3_case_promotion()

⚙️  STAGE 3: CASE PROMOTION

📥 Input: 21 risk profiles from Stage 2


2026-03-07 21:01:29.161 | INFO     | services.case_promotion:promote_profiles_to_cases:58 - Starting case promotion: min_priority=low, limit=None
2026-03-07 21:01:29.234 | INFO     | services.case_promotion:promote_profiles_to_cases:84 - Retrieved 21 risk profiles
2026-03-07 21:01:29.235 | INFO     | services.case_promotion:promote_profiles_to_cases:93 - Filtered to 21 profiles meeting priority threshold
2026-03-07 21:01:29.236 | DEBUG    | services.case_promotion:_process_single_profile:166 - Processing profile for user: mileycyrus



🚀 Running case promotion...
   • Reading from: instagram_scraper.case_risk_profiles
   • Writing to: dellinnovate.scs_cases


2026-03-07 21:01:29.455 | INFO     | services.case_promotion:_create_new_case:255 - Created case CASE_2026_022 with priority=low, category=Low
2026-03-07 21:01:29.599 | DEBUG    | services.case_promotion:_add_case_history:329 - Added history entry for case CASE_2026_022
2026-03-07 21:01:30.024 | INFO     | services.case_promotion:_create_checklist_items:370 - Created 4 checklist items for case CASE_2026_022
2026-03-07 21:01:30.025 | INFO     | services.case_promotion:_process_single_profile:182 - Created new case CASE_2026_022 for user mileycyrus
2026-03-07 21:01:30.025 | DEBUG    | services.case_promotion:_process_single_profile:166 - Processing profile for user: crime101film
2026-03-07 21:01:30.247 | INFO     | services.case_promotion:_create_new_case:255 - Created case CASE_2026_023 with priority=low, category=Low
2026-03-07 21:01:30.393 | DEBUG    | services.case_promotion:_add_case_history:329 - Added history entry for case CASE_2026_023
2026-03-07 21:01:30.819 | INFO     | servic

## Step 5: Final Status Check

Verify the complete pipeline execution and compare before/after state.

In [ ]:
async def check_final_state():
    """Show final state and pipeline summary"""
    
    source_db = MongoDB.get_source_db()
    db = MongoDB.get_db()
    
    print("=" * 70)
    print("📊 FINAL DATABASE STATE")
    print("=" * 70)
    
    print("\n📥 SOURCE & ANALYTICS (instagram_scraper):")
    posts_count = await source_db.instagram_posts.count_documents({})
    users_count = await source_db.instagram_users.count_documents({})
    signals_count = await source_db.text_units_signals.count_documents({})
    profiles_count = await source_db.case_risk_profiles.count_documents({})
    print(f"  Posts: {posts_count:,}")
    print(f"  Users: {users_count:,}")
    print(f"  NLP Signals: {signals_count:,} ✅")
    print(f"  Risk Profiles: {profiles_count:,} ✅")
    
    print("\n⚙️  OPERATIONAL (dellinnovate):")
    cases_count = await db.scs_cases.count_documents({})
    history_count = await db.scs_case_history.count_documents({})
    checklist_count = await db.scs_checklist.count_documents({})
    print(f"  Cases: {cases_count:,} ✅")
    print(f"  Case History: {history_count:,} ✅")
    print(f"  Checklist Items: {checklist_count:,} ✅")
    
    # Compare with initial state
    print("\n📈 PIPELINE CHANGES:")
    signals_added = signals_count - initial_state['signals_before']
    profiles_added = profiles_count - initial_state['profiles_before']
    cases_added = cases_count - initial_state['cases_before']
    history_added = history_count - initial_state['history_before']
    checklist_added = checklist_count - initial_state['checklist_before']
    
    print(f"  Stage 1 Output: +{signals_added:,} signals")
    print(f"  Stage 2 Output: +{profiles_added:,} profiles")
    print(f"  Stage 3 Output: +{cases_added:,} cases, +{history_added:,} history, +{checklist_added:,} checklist items")
    
    # Show priority distribution
    print("\n📊 Case Priority Distribution:")
    pipeline = [
        {"$group": {"_id": "$priority", "count": {"$sum": 1}, "avg_risk": {"$avg": "$current_risk_score"}}},
        {"$sort": {"_id": 1}}
    ]
    distribution = await db.scs_cases.aggregate(pipeline).to_list(length=10)
    
    for item in distribution:
        count = item['count']
        avg_risk = item['avg_risk']
        bar = "█" * int(count * 2)
        print(f"  {item['_id'].upper():8s}: {count:3d} cases (avg risk: {avg_risk:.1f}) {bar}")
    
    print("\n" + "=" * 70)
    print("✅ PIPELINE COMPLETE!")
    print("=" * 70)
    print("\n🎯 Next Steps:")
    print("   • Open dashboard to view cases")
    print("   • Assign cases to youth workers")
    print("   • Begin intervention workflows")
    print("   • Schedule automated daily runs")
    print("\n" + "=" * 70)

await check_final_state()

📊 FINAL DATABASE STATE

📥 SOURCE & ANALYTICS (instagram_scraper):
  Posts: 164
  Users: 11
  NLP Signals: 225 ✅
  Risk Profiles: 21 ✅

⚙️  OPERATIONAL (dellinnovate):
  Cases: 21 ✅
  Case History: 21 ✅
  Checklist Items: 84 ✅

📈 PIPELINE CHANGES:
  Stage 1 Output: +0 signals
  Stage 2 Output: +0 profiles
  Stage 3 Output: +11 cases, +5 history, +39 checklist items

📊 Case Priority Distribution:
  MEDIUM  :  21 cases (avg risk: 50.0) ██████████████████████████████████████████

✅ PIPELINE COMPLETE!

🎯 Next Steps:
   • Open dashboard to view cases
   • Assign cases to youth workers
   • Begin intervention workflows
   • Schedule automated daily runs



## Step 6: View Sample Cases (Optional)

Display created cases with full details.

In [ ]:
async def show_sample_cases(limit=5):
    """Display sample cases with full details"""
    
    db = MongoDB.get_db()
    
    cases = await db.scs_cases.find().sort("current_risk_score", -1).limit(limit).to_list(length=limit)
    
    print("=" * 70)
    print(f"📋 Sample Cases (Top {limit} by Risk Score)")
    print("=" * 70)
    
    for i, case in enumerate(cases, 1):
        print(f"\n{i}. Case ID: {case['case_id']}")
        print(f"   User: {case['user_id']}")
        print(f"   Priority: {case['priority'].upper()} ⭐")
        print(f"   Risk Score: {case['current_risk_score']:.2f}/100")
        print(f"   Category: {case['category']}")
        print(f"   Status: {case['case_status']}")
        print(f"   Work Status: {case['work_status']}")
        print(f"   Created: {case['created_at']}")
        print(f"   AI Explanation: {case['ai_explanation'][:120]}...")
        
        # Show checklist status
        checklist_items = await db.scs_checklist.find({"case_id": case['case_id']}).to_list(length=10)
        completed = sum(1 for item in checklist_items if item['completed'])
        total = len(checklist_items)
        print(f"   Checklist: {completed}/{total} completed")
    
    print("\n" + "=" * 70)

await show_sample_cases(limit=5)

📋 Sample Cases (Top 5 by Risk Score)

1. Case ID: CASE_2026_004
   User: mohammed.usrof
   Priority: MEDIUM ⭐
   Risk Score: 50.00/100
   Category: Unknown
   Status: unassigned
   Work Status: not_started
   Created: 2026-03-07 12:54:24.857000
   AI Explanation: Risk detected through automated NLP analysis of social media content....
   Checklist: 0/4 completed

2. Case ID: CASE_2026_005
   User: fallontonight
   Priority: MEDIUM ⭐
   Risk Score: 50.00/100
   Category: Unknown
   Status: unassigned
   Work Status: not_started
   Created: 2026-03-07 12:54:24.857000
   AI Explanation: Risk detected through automated NLP analysis of social media content....
   Checklist: 0/4 completed

3. Case ID: CASE_2026_002
   User: disneyplus
   Priority: MEDIUM ⭐
   Risk Score: 50.00/100
   Category: Unknown
   Status: unassigned
   Work Status: not_started
   Created: 2026-03-07 12:54:24.857000
   AI Explanation: Risk detected through automated NLP analysis of social media content....
   Checklist

## Summary & Automation

### ✅ Pipeline Execution Complete

You've successfully run the entire pipeline:
1. ✅ **Stage 1**: NLP signal extraction from posts
2. ✅ **Stage 2**: Behavioral feature engineering & risk profiling
3. ✅ **Stage 3**: Case promotion to operational system

### 🤖 Automate This Pipeline

To run this automatically on a schedule:

```bash
# Start the automated scheduler (runs daily at 2 AM)
cd backend
python pipeline_scheduler.py
```

Or use the REST API:

```bash
# Trigger full pipeline execution
curl -X POST http://localhost:8000/api/analytics/run-full-pipeline
```

### 📚 Documentation

- [ANALYTICS_README.md](../ANALYTICS_README.md) - Analytics pipeline overview
- [IMPLEMENTATION_SUMMARY.md](../backend/IMPLEMENTATION_SUMMARY.md) - Backend architecture
- [CASE_PROMOTION_README.md](../backend/CASE_PROMOTION_README.md) - Case promotion details